# Stride sweep experiments (Burgers)

This notebook runs a sweep over temporal subsampling `stride_t ? {1,5,7,10}` for two spatial settings `stride_x ? {5,10}`.

**Controls**
- `steps=8000`, `noise=0.7`, `nu=0.02`, `lr=0.001`, `batch_size=1000`
- `lam_pde=0.5`, `lam_tv=0`, `lam_reg=0`, `lam_data=10`
- `selected_derivs=('u','u_x','u_xx')`

**Outputs per run**
- Final `loss_data`, `loss_pde`
- Linear coefficients from `eql.readout.weight`: `w_u`, `w_ux`, `w_uxx`, `w_prod`
- Nonlinear `u*u_x` contributions from unsymmetrized effective matrix: `M01 = M[0,1]`, `M10 = M[1,0]` where `M = eql.effective_quadratic_matrix(symmetrize=False)`.


In [ ]:
import sys

sys.path.append('..')  # add project root

import numpy as np
import torch
import matplotlib.pyplot as plt

from prog import mlps, featlib, trainer, hlprs
import Datasets.matconv as mc

SEED = 1432
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')


In [ ]:
# Controls

steps = 8000
log_every = 1000

noise = 0.7
nu = 0.02

lr = 1e-3
batch_size = 1000

lam_pde = 0.5
lam_tv = 0.0
lam_reg = 0.0
lam_data = 10.0

selected_derivs = ('u','u_x','u_xx')

# Sweeps
stride_t_values = [1, 5, 7, 10]
stride_x_values = [5, 10]

# Dataset partitioning controls
part_num = 1
which_part = 1


In [ ]:
def run_once(*, stride_t: int, stride_x: int):
    # Repro per run
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # --- dataset ---
    partitions = mc.build_dataset_from_burgers(
        noise_level=noise,
        nu=nu,
        stride_t=stride_t,
        stride_x=stride_x,
        seed=SEED,
        quantile_splits=part_num,
        return_partitions=True,
    )
    key = [k for k in partitions if k.startswith(f'Q{which_part}:')][0]
    t_np, x_np, y_np, y_noisy_np, _N = partitions[key]

    t_torch       = torch.from_numpy(t_np).to(device)
    x_torch       = torch.from_numpy(x_np).to(device)
    y_clean_torch = torch.from_numpy(y_np).to(device)
    y_noisy_torch = torch.from_numpy(y_noisy_np).to(device)

    # --- models ---
    u_model = mlps.SimpleMLP(n_layers=4, hidden_size=64, act=mlps.Sin)
    symnet  = mlps.EQL(in_dim=len(selected_derivs), prod_dim=2, num_layers=1, bias=False)

    train_config = trainer.TrainerConfig(
        lr=lr,
        lambda_pde=lam_pde,
        lambda_reg=lam_reg,
        lambda_tv=lam_tv,
        lambda_data=lam_data,
        selected_derivs=selected_derivs,
        device=device,
    )

    ft = featlib.FeatureTensor(selected_derivs, normalize=False)
    feature_builder = ft.build

    train = trainer.PDETrainer(
        u_model=u_model,
        v_model=symnet,
        cfg=train_config,
        feature_builder=feature_builder,
    )

    # --- train ---
    loss_dat = []
    loss_pde = []

    for i in range(steps):
        t, x, u_noisy, u_clean = hlprs.make_batch(
            batch_size=batch_size,
            t_torch=t_torch,
            x_torch=x_torch,
            y_clean=y_clean_torch,
            y_noisy=y_noisy_torch,
        )
        out = train.step(t=t, x=x, u_noisy=u_noisy, u_clean=u_clean)
        loss_dat.append(out['loss_data'])
        loss_pde.append(out['loss_pde'])

        if log_every and i % log_every == 0:
            print(f"[stride_x={stride_x} stride_t={stride_t}] step {i}: data={out['loss_data']:.6e} pde={out['loss_pde']:.6e}")

    # --- coefficients ---
    w = symnet.readout.weight.detach().cpu().numpy().reshape(-1)
    M = symnet.effective_quadratic_matrix(symmetrize=False).detach().cpu().numpy()

    result = {
        'stride_x': int(stride_x),
        'stride_t': int(stride_t),
        'loss_data_final': float(loss_dat[-1]),
        'loss_pde_final': float(loss_pde[-1]),
        'w_u': float(w[0]),
        'w_ux': float(w[1]),
        'w_uxx': float(w[2]),
        'w_prod': float(w[-1]),
        'M01': float(M[0, 1]),
        'M10': float(M[1, 0]),
        # keep full objects for inspection
        'readout_weight': w,
        'M_eff': M,
        'loss_data_curve': loss_dat,
        'loss_pde_curve': loss_pde,
    }

    return result


In [ ]:
# Run sweep (8 runs)

results = []
for stride_x in stride_x_values:
    for stride_t in stride_t_values:
        print('
==============================')
        print('Running stride_x=', stride_x, 'stride_t=', stride_t)
        print('==============================')
        results.append(run_once(stride_t=stride_t, stride_x=stride_x))


In [ ]:
# Build + display tables

try:
    import pandas as pd
except ImportError:
    pd = None

cols = [
    'stride_x','stride_t',
    'loss_data_final','loss_pde_final',
    'w_u','w_ux','w_uxx','w_prod',
    'M01','M10'
]

rows = [{k: r[k] for k in cols} for r in results]

if pd is not None:
    df = pd.DataFrame(rows).sort_values(['stride_x','stride_t']).reset_index(drop=True)
    display(df)
    print('
Markdown table (paste into email):')
    print(df.to_markdown(index=False))

    print('
--- By stride_x ---')
    for sx in sorted(df['stride_x'].unique()):
        sub = df[df['stride_x'] == sx].copy()
        print(f"
stride_x = {sx}")
        print(sub.to_markdown(index=False))
else:
    # fallback: simple print
    print(rows)
